In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sqlite3
import csv

### Setup

In [5]:
SOURCE_DATA_PATH = "./data/"
EL_CONX_LIST_SEM_FILTERED = "elative_construction_phrase_list.csv"
EL_CONX_LIST_UNFILTERED = "elative_construction_not_material_attribute_phrase_list.csv"

TR_SOURCE_DATA_PATH = "C:/Users/liivas/Downloads/Töö/estnltk_projekt_2025/transaktsioonid/source_data/"
TR_DB = "v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310.db"

EL_CONX_PHRASES = "elative_conx_phrases.db"
RESULT = "elative_conx_conflicts.db"

### Methods

In [ ]:
def find_consecutive_phrase(cursor, word1, word2):
    cursor.execute("""
        CREATE TABLE result.phrase_matches_consecutive AS
        SELECT
            w1.head_id,
            w1.id,
        FROM 
            phrases AS p
        INNER JOIN 
            tr.transaction_row AS w1
        ON 
            w1.form = p.member1
        INNER JOIN
            tr.transaction_row AS w2
        ON
            w2.head_id = w1.head_id
        AND
            w2.form = p.member2
        AND 
            w2.loc = w1.loc + 1
    """)

    conn.commit()

In [ ]:
def find_ordered_phrase(cursor, word1, word2):
    cursor.execute("""
        CREATE TABLE result.phrase_matches_consecutive AS
        SELECT
            w1.head_id,
            w1.id,
        FROM 
            phrases AS p
        INNER JOIN 
            tr.transaction_row AS w1
        ON 
            w1.form = p.member1
        INNER JOIN
            tr.transaction_row AS w2
        ON
            w2.head_id = w1.head_id
        AND
            w2.form = p.member2
        AND 
            w2.loc > w1.loc + 1
    """)

    conn.commit()

### Saving CSV-files into database

In [6]:
conn = sqlite3.connect(f"{SOURCE_DATA_PATH}{EL_CONX_PHRASES}")
cur = conn.cursor()

cur.execute("""
    CREATE TABLE IF NOT EXISTS phrases_sem_filtered (
        member1 TEXT,
        member2 TEXT,
        freq INTEGER
    )
""")

with open(f"{SOURCE_DATA_PATH}{EL_CONX_LIST_SEM_FILTERED}", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)

    rows = [
        (row["member1"], row["member2"], row["freq"])
        for row in reader
    ]

cur.executemany("""
    INSERT INTO phrases_sem_filtered (member1, member2, freq)
    VALUES (?, ?, ?)
""", rows)

conn.commit()

cur.execute("""
    CREATE TABLE IF NOT EXISTS phrases_other (
        member1 TEXT,
        member2 TEXT,
        freq INTEGER
    )
""")

with open(f"{SOURCE_DATA_PATH}{EL_CONX_LIST_UNFILTERED}", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)

    rows = [
        (row["member1"], row["member2"], row["freq"])
        for row in reader
    ]

cur.executemany("""
    INSERT INTO phrases_other (member1, member2, freq)
    VALUES (?, ?, ?)
""", rows)

conn.commit()

conn.close()

### Finding conflicts from transactions database

In [7]:
con = sqlite3.connect(f"{TR_SOURCE_DATA_PATH}{TR_DB}")

cur = con.cursor()

cur.execute("""
    CREATE INDEX transaction_form_loc
    ON transaction_row(form, loc)
""")

con.close()

### I Consecutive phrases

Semantically filtered phrases:

In [11]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{EL_CONX_PHRASES}")

cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{TR_SOURCE_DATA_PATH}{TR_DB}" AS tr')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT}" AS result')

cur.execute("""
DROP TABLE IF EXISTS result.phrase_conflicts_filtered_consecutive
""")

cur.execute("""
    CREATE TABLE result.phrase_conflicts_filtered_consecutive AS
    SELECT
        w1.head_id,
        w1.id
    FROM 
        phrases_sem_filtered AS p
    INNER JOIN 
        tr.transaction_row AS w1
    ON 
        w1.form = p.member1
    INNER JOIN
        tr.transaction_row AS w2
    ON
        w2.head_id = w1.head_id
    WHERE
        w2.form = p.member2
    AND 
        w2.loc = w1.loc + 1
""")

con.close()

Other phrases:

In [12]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{EL_CONX_PHRASES}")

cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{TR_SOURCE_DATA_PATH}{TR_DB}" AS tr')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT}" AS result')

cur.execute("""
DROP TABLE IF EXISTS result.phrase_conflicts_other_consecutive
""")

cur.execute("""
    CREATE TABLE result.phrase_conflicts_other_consecutive AS
    SELECT
        w1.head_id,
        w1.id
    FROM 
        phrases_other AS p
    INNER JOIN 
        tr.transaction_row AS w1
    ON 
        w1.form = p.member1
    INNER JOIN
        tr.transaction_row AS w2
    ON
        w2.head_id = w1.head_id
    WHERE
        w2.form = p.member2
    AND 
        w2.loc = w1.loc + 1
""")

con.close()

### II Non-consecutive phrases

In [ ]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{EL_CONX_PHRASES}")

cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{TR_SOURCE_DATA_PATH}{TR_DB}" AS tr')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT}" AS result')

cur.execute("""
DROP TABLE IF EXISTS result.phrase_conflicts_filtered_ordered
""")

cur.execute("""
    CREATE TABLE result.phrase_conflicts_filtered_ordered AS
    SELECT
        w1.head_id,
        w1.id
    FROM 
        phrases_sem_filtered AS p
    INNER JOIN 
        tr.transaction_row AS w1
    ON 
        w1.form = p.member1
    INNER JOIN
        tr.transaction_row AS w2
    ON
        w2.head_id = w1.head_id
    WHERE
        w2.form = p.member2
    AND 
        w2.loc > w1.loc + 1
""")

con.close()

: 

In [ ]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{EL_CONX_PHRASES}")

cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{TR_SOURCE_DATA_PATH}{TR_DB}" AS tr')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT}" AS result')

cur.execute("""
DROP TABLE IF EXISTS result.phrase_conflicts_other_ordered
""")

cur.execute("""
    CREATE TABLE result.phrase_conflicts_other_ordered AS
    SELECT
        w1.head_id,
        w1.id
    FROM 
        phrases_other AS p
    INNER JOIN 
        tr.transaction_row AS w1
    ON 
        w1.form = p.member1
    INNER JOIN
        tr.transaction_row AS w2
    ON
        w2.head_id = w1.head_id
    WHERE
        w2.form = p.member2
    AND 
        w2.loc > w1.loc + 1
""")

con.close()